### Anomaly detection model achitecture and hyperparameter tuning 
No we are ready to define GAN based anomaly detection model architecture and perform hyperparameter tuning using maggy. For more details about this model refer to https://arxiv.org/pdf/1905.11034.pdf.
![Training Dataset](./images/maggy_hp.png)

## Install GAN based anomaly detection model from the git repository. 
![Incremental Feature Engineering](./images/intall_python_lib_from_git.png)

In [1]:
# Setup for local execution
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import roc_auc_score
import itertools
import random

# Define paths
BASE_PATH = os.path.dirname(os.path.abspath("__file__"))
TRAINING_DATA_PATH = os.path.join(BASE_PATH, "training_data")
RESOURCES_PATH = os.path.join(BASE_PATH, "Resources")
GAN_DATA_PATH = os.path.join(TRAINING_DATA_PATH, "gan")

print(f"TensorFlow version: {tf.__version__}")
print(f"Data path: {GAN_DATA_PATH}")

2026-02-02 14:32:44.094302: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-02 14:32:44.138201: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-02 14:32:45.249614: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow version: 2.20.0
Data path: /home/adnoman/projects/aml_gan/AMLend2end/training_data/gan


In [2]:
# Load embeddings hyperparameters
emb_hp_path = os.path.join(RESOURCES_PATH, "embeddings_best_hp.json")
with open(emb_hp_path, 'r') as f:
    emb_best_hp = json.load(f)
    
input_dim = emb_best_hp['emb_size']
print(f"Input dimension (embedding size): {input_dim}")

Input dimension (embedding size): 32


In [3]:
# Load training data
X_train = np.load(os.path.join(GAN_DATA_PATH, "X_train.npy"))
y_train = np.load(os.path.join(GAN_DATA_PATH, "y_train.npy"))
X_eval = np.load(os.path.join(GAN_DATA_PATH, "X_eval.npy"))
y_eval = np.load(os.path.join(GAN_DATA_PATH, "y_eval.npy"))

print(f"Training data: {X_train.shape}")
print(f"Evaluation data: {X_eval.shape}")

Training data: (5224, 32)
Evaluation data: (2123, 32)


### Define hopsworks experiments wrapper function and put all the training logic there. 

In [4]:
# Simplified Autoencoder-based Anomaly Detector
# (Replaces adversarialaml GAN - similar concept, simpler implementation)

def build_autoencoder(input_dim, latent_dim, n_layers, activation, dropout_rate, learning_rate):
    """Build an autoencoder for anomaly detection."""
    
    # Encoder
    encoder_input = layers.Input(shape=(input_dim,))
    x = encoder_input
    
    units = input_dim
    for i in range(n_layers):
        units = max(units // 2, latent_dim)
        x = layers.Dense(units, activation=activation)(x)
        if dropout_rate > 0:
            x = layers.Dropout(dropout_rate)(x)
    
    latent = layers.Dense(latent_dim, activation=activation, name='latent')(x)
    
    # Decoder
    x = latent
    units = latent_dim
    for i in range(n_layers):
        units = min(units * 2, input_dim)
        x = layers.Dense(units, activation=activation)(x)
        if dropout_rate > 0:
            x = layers.Dropout(dropout_rate)(x)
    
    decoder_output = layers.Dense(input_dim, activation='linear')(x)
    
    # Full autoencoder
    autoencoder = keras.Model(encoder_input, decoder_output, name='autoencoder')
    autoencoder.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse'
    )
    
    return autoencoder

def compute_anomaly_score(model, X):
    """Compute reconstruction error as anomaly score."""
    X_pred = model.predict(X, verbose=0)
    mse = np.mean(np.square(X - X_pred), axis=1)
    return mse

## The searchspace can be instantiated with parameters

In [5]:
# Define hyperparameter search space (simplified)
search_space = {
    'latent_dim': [8, 16],
    'n_layers': [2, 3],
    'activation': ['relu', 'tanh'],
    'dropout_rate': [0.0, 0.1],
    'learning_rate': [0.001, 0.0001]
}

# Generate combinations
param_combinations = list(itertools.product(
    search_space['latent_dim'],
    search_space['n_layers'],
    search_space['activation'],
    search_space['dropout_rate'],
    search_space['learning_rate']
))

print(f"Total combinations: {len(param_combinations)}")

Total combinations: 32


## Use above experiments wrapper function to conduct hops training experiments.

In [6]:
# Run hyperparameter search (replaces Maggy)
num_trials = 2
random.seed(42)
trial_combinations = random.sample(param_combinations, min(num_trials, len(param_combinations)))

results = []
best_auc = 0
best_hp = None
best_model = None

print(f"Running {len(trial_combinations)} trials...")
print("=" * 50)

for i, (latent_dim, n_layers, activation, dropout_rate, learning_rate) in enumerate(trial_combinations):
    print(f"\n--- Trial {i+1}/{len(trial_combinations)} ---")
    hp = {
        'latent_dim': latent_dim,
        'n_layers': n_layers,
        'activation': activation,
        'dropout_rate': dropout_rate,
        'learning_rate': learning_rate
    }
    print(f"Hyperparameters: {hp}")
    
    try:
        # Build and train model
        model = build_autoencoder(input_dim, latent_dim, n_layers, activation, dropout_rate, learning_rate)
        
        history = model.fit(
            X_train, X_train,
            epochs=10,
            batch_size=32,
            validation_split=0.1,
            verbose=0
        )
        
        # Evaluate using AUC on anomaly detection
        anomaly_scores = compute_anomaly_score(model, X_eval)
        auc = roc_auc_score(y_eval, anomaly_scores)
        
        print(f"Loss: {history.history['loss'][-1]:.4f}, AUC: {auc:.4f}")
        
        results.append({'hp': hp, 'auc': auc, 'loss': history.history['loss'][-1]})
        
        if auc > best_auc:
            best_auc = auc
            best_hp = hp
            best_model = model
            
    except Exception as e:
        print(f"Trial failed: {e}")
        results.append({'hp': hp, 'auc': None, 'error': str(e)})

print("\n" + "=" * 50)
print("------ Hyperparameter Search Results ------")
print(f"BEST combination {best_hp}")
print(f"BEST AUC: {best_auc:.4f}")
print("=" * 50)

Running 2 trials...

--- Trial 1/2 ---
Hyperparameters: {'latent_dim': 8, 'n_layers': 2, 'activation': 'tanh', 'dropout_rate': 0.1, 'learning_rate': 0.0001}


I0000 00:00:1770024779.294222   36507 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9511 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4080 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
2026-02-02 14:33:00.639028: I external/local_xla/xla/service/service.cc:163] XLA service 0x74c9a000d170 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-02 14:33:00.639063: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4080 Laptop GPU, Compute Capability 8.9
2026-02-02 14:33:00.660596: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-02 14:33:00.813738: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91801
I0000 00:00:1770024782.584465   37135 device_compiler.h:196] Compiled cluster using XLA!  This

Loss: 0.0003, AUC: 0.5137

--- Trial 2/2 ---
Hyperparameters: {'latent_dim': 8, 'n_layers': 2, 'activation': 'relu', 'dropout_rate': 0.0, 'learning_rate': 0.0001}
Loss: 0.0003, AUC: 0.5251

------ Hyperparameter Search Results ------
BEST combination {'latent_dim': 8, 'n_layers': 2, 'activation': 'relu', 'dropout_rate': 0.0, 'learning_rate': 0.0001}
BEST AUC: 0.5251


In [7]:
# Save best hyperparameters
gan_hp_path = os.path.join(RESOURCES_PATH, "gan_best_hp.json")
with open(gan_hp_path, 'w') as f:
    json.dump(best_hp, f, indent=2)

print(f"Saved best hyperparameters to: {gan_hp_path}")
print(f"Best HP: {best_hp}")

Saved best hyperparameters to: /home/adnoman/projects/aml_gan/AMLend2end/Resources/gan_best_hp.json
Best HP: {'latent_dim': 8, 'n_layers': 2, 'activation': 'relu', 'dropout_rate': 0.0, 'learning_rate': 0.0001}


### Managing experiments
Experiments service provides a unified view of all the experiments run using the `experiment` module.
<br>
As demonstrated in the gif it provides general information about the experiment and the resulting metric. Experiments can be visualized meanwhile or after training in a TensorBoard.
<br>
<br>
![Image7-Monitor.png](./images/experiments.png)

In [ ]:
# Done!